In [47]:
import os
import re
import xml.etree.cElementTree as ET
from cltk.stop.arabic.stopword_filter import stopwords_filter
from cltk.corpus.arabic.alphabet import LETTERS
from nltk.stem.isri import ISRIStemmer
stemmer = ISRIStemmer()
nums='۰-۹'
letters=''.join(LETTERS)
reg='[^0-9A-Za-z'+str(nums)+str(letters)+']+'

def preprocess(text):
    txtList=stopwords_filter(text)
    l=[]
    for piece in txtList:
        piece=re.sub(reg, ' ', piece)
        piece=piece.split(' ')
        for word in piece:
            word=stemmer.stem(word)
            word=stopwords_filter(word)
            if len(word)!= 0:
                l.append(word[0])
    return l

def create_invertedIndex(docsTokens):
    invIndex={}
    for d in range(0, len(docsTokens)):
        for token in docsTokens[d]:
            word=token[0]
            tag=token[1]
            if token[0] not in invIndex:
                invIndex[word]={d:[1,tag]}
            elif d not in invIndex[word]:
                invIndex[word][d] = [1,tag]
            else:
                invIndex[word][d][0] = invIndex[word][d][0] + 1
    return invIndex


    
from nltk.tag import StanfordPOSTagger
from nltk import word_tokenize
jar = '/home/mosab/Desktop/IR/Ar/Project/stanford-postagger-full-2018-02-27/stanford-postagger.jar'
model = '/home/mosab/Desktop/IR/Ar/Project/stanford-postagger-full-2018-02-27/models/arabic.tagger'
pos_tagger = StanfordPOSTagger(model, jar, encoding='utf8')


class docInfo:
    def __init__(self, docName,docLen):
        self.docName = docName
        self.docLen = docLen
    def __repr__(self):
        return __str__(self)
    def __str__(self):
        return "Name = "+str(self.docName)+"\nLength = "+str(self.docLen)
    

def create_queryTFs(queryTokens):
    qTFs={}
    for token in queryTokens:
        word=token[0]
        tag=token[1]
        if word not in qTFs:
            qTFs[word]=[[1,tag]]
        else:
            l=qTFs[word]
            l=[x[1] for x in l]
            try:
                index=l.index(tag)
                qTFs[word][index][0] = qTFs[word][index][0] + 1
            except:
                qTFs[word].append([1,tag])
            
    return qTFs


import pickle
with open("generated/documentsNames.txt", "rb") as fp:
    documentNames = pickle.load(fp)
    
with open("generated/documentsInfo.txt", "rb") as fp:
    documentsInfo = pickle.load(fp)

'''
with open("generated/docsTokens.txt", "rb") as fp:
    docsTokens = pickle.load(fp)
'''

with open("generated/invertedIndex.txt", "rb") as fp:
    invertedIndex = pickle.load(fp)

print("==================================Loading is Done==================================")

==================================Loading is Done==================================


In [48]:
class queryInfo:
    def __init__(self, queryTitle,queryTFs):
        self.queryTitle = queryTitle
        self.queryTFs = queryTFs
    def __repr__(self):
        return __str__(self)
    def __str__(self):
        return "Title = "+str(self.queryTitle)+"\nTFs = "+str(self.queryTFs)
        
import xml.etree.cElementTree as ET
queries={}
with open("/media/mosab/D Drive/lessons/PhD/Semester2/Information Retrieval/Ass2/Ar/Arabic Corpus/q_txt_gz.txt") as f:
    lines = f.read()
matches = re.findall(r"#\s*q\s*([\d]+)\s*=\s*#sum\(\s*([\w\W]*?)\s*\)\s*;", lines)
for match in matches:
    queryNum = int(match[0])
    queryTitle = match[1]
    
    TEXT= queryTitle
    TEXT = pos_tagger.tag(word_tokenize(TEXT))
    
    queryTokens=[]
    for token in TEXT:
        temp=[]
        wordTag = token[1].split('/')
        if(len(wordTag)<=1):
            wordTag = token[0].split('/')
        if(len(wordTag)<=1):
            continue
        l=preprocess(wordTag[0])
        if len(l)==0:
            continue
        temp.append(l[0])
        temp.append(wordTag[1])
        queryTokens.append(temp)
        
    queries[queryNum] = queryInfo(queryTitle, create_queryTFs(queryTokens))

print("==================================Loading Queries is Done==================================")

==================================Loading Queries is Done==================================


In [49]:
answers={}
with open("/media/mosab/D Drive/lessons/PhD/Semester2/Information Retrieval/Ass2/Ar/Arabic Corpus/qrels25_txt_gz.txt") as f:
    lines = f.readlines()

for i in range(0, len(lines)-1, 2):
    tokens=lines[i].split()
    if int(tokens[0]) not in answers:
        answers[int(tokens[0])]=[documentNames.index(int(tokens[2]))]
    else:
        answers[int(tokens[0])].append(documentNames.index(int(tokens[2])))
       
    answers[int(tokens[0])].sort()

print("==================================Loading Answers is Done==================================")

==================================Loading Answers is Done==================================


In [50]:
def lengthIntersection(arr1, arr2, m, n):
    i,j,l = 0,0,0
    while i < m and j < n:
        if arr1[i] < arr2[j]:
            i += 1
        elif arr2[j] < arr1[i]:
            j+= 1
        else:
            l=l+1
            j += 1
            i += 1
    return l

def get_r(qNum,t,tag):
    '''if t not in invertedIndex:
        indices=[]
    else:
    '''
    #indices = list(invertedIndex[t].keys())
    
    indices=[]
    for d in invertedIndex[t]:
        l=[x[1] for x in invertedIndex[t][d]]
        if(tag in l):
            indices.append(d)
            
    indices.sort()
    return lengthIntersection(answers[qNum] , indices , len(answers[qNum]) , len(indices))

def get_R(qNum):
    return len(answers[qNum])

def get_df_t(t,tag):
    '''if t not in invertedIndex:
        return 0
    else:
    '''
    df_t=0
    for d in invertedIndex[t]:
        l=[x[1] for x in invertedIndex[t][d]]
        if(tag in l):
            df_t=df_t+1
            
    return df_t

def get_n():
    return len(documentsInfo)

def get_tf_d(d,t,tag):
    '''if t not in invertedIndex:
        return 0
    elif d not in invertedIndex[t]:
        return 0
    else:
    '''
    for x in invertedIndex[t][d]:
        if(tag==x[1]):
            return x[0]
    return 0

def get_L_d(d):
    return documentsInfo[d].docLen

def get_L_avg():
    l=0
    for i in range(0, len(documentsInfo)):
        l=l+documentsInfo[i].docLen
    return l/get_n()

def get_tf_q(qNum,t):
    return queries[qNum].queryTFs[t]


import pylab as pl

k1=1.0
b=0.6
k3=0.0

λc_L=pl.frange(0.1,2,0.1)
k=10
ϴ=0.5


from gensim.models import KeyedVectors
model = KeyedVectors.load_word2vec_format("/home/mosab/Desktop/IR/Ar/Project/word2vec.model.txt", binary = False)
'''
def Sc(w):
    w=preprocess(w)
    return model.most_similar(w,topn=k)

def Ac(w,w_):
    w=preprocess(w)
    word_similarity_pairs=model.most_similar(w,topn=k)
    i=0
    similarWords=[]
    for word_similarity_pair in word_similarity_pairs:
        if word_similarity_pair[1]>ϴ:
            similarWords.append(word_similarity_pair[0])
'''   


import math

n=get_n()
L_avg=get_L_avg()
def okapiBM25(qNum,d,k1,b,k3,λc):
    R = get_R(qNum)
    L_d=get_L_d(d)
    score=0
        
    for t in queries[qNum].queryTFs.keys():
        word_similarity_pairs=model.most_similar(t,topn=k-1)
        word_similarity_pairs.insert(0,(t,1.0))
        pos_t = invertedIndex[t]
        i=1 # first word is t itself so skip it
        sumCos=1 #because t has weight 1
        while i<len(word_similarity_pairs):
            if (word_similarity_pairs[i][1]>ϴ) or ():
                sumCos=sumCos+word_similarity_pairs[i][1]
                i=i+1
            else:
                del word_similarity_pairs[i:]
        
        tf_qs=get_tf_q(qNum,t)
        for tf_q_ in tf_qs:
            tf_q=tf_q_[0]
            tag=tf_q_[1]
            A_part = ( (k3+1)*tf_q)/( k3+tf_q)
            B_part=0
            for pair in word_similarity_pairs:
                similar = pair[0]
                similarity = λc*pair[1]/sumCos
                if(similar==t):
                    similarity=1

                if similar not in invertedIndex:
                    continue
                elif d not in invertedIndex[similar]:
                    continue
                else:
                    l=invertedIndex[similar][d]
                    l=[x[1] for x in l]
                    try:
                        index=l.index(tag)
                    except:
                        continue

                r=get_r(qNum,similar, tag)
                df_t=get_df_t(similar, tag)
                tf_d=get_tf_d(d,similar, tag)
                B_part1 = math.log(  ((r+0.5) / (R-r+0.5))  /  ((df_t-r+0.5)  /  (n-df_t-R+r+0.5))  )
                #try:
                B_part2 = ( (k1+1)*tf_d )/( k1*(1-b+b*(L_d/L_avg))+tf_d )
                '''except:
                    print("k1=" +str(k1)+"tf_d=" +str(tf_d)+"L_d=" +str(L_d)+"L_avg=" +str(L_avg))
                    exit()'''
                B_part = B_part+ similarity*B_part1*B_part2
            score=score+A_part*B_part
    return score

def okapiBM25_Docs(qNum,k1,b,k3,λc):
    docsScore=[]
    for d in range(0,n):
        score = okapiBM25(qNum,d,k1,b,k3,λc)
        docsScore.append(score)
    return docsScore

def call_OkapiBM25_Docs(q,qNum,k1,b,k3,i):
    documentsScore=okapiBM25_Docs(qNum,k1,b,k3,λc_L[i])
    q.put(documentsScore)
    

In [51]:
from multiprocessing import Process, Queue

def initializeDocumentsScores():
    documentsScores=[]
    for i in range(0, len(queries.keys())):
        documentsScores.append([])
    return documentsScores

def get_Q():
    return len(queries)

import numpy
def AP(documentsScore, qNum):
    relevantDocs = answers[qNum]
    R=get_R(qNum)
    retrievedDocuments=numpy.argsort(documentsScore).tolist()[::-1]#returns the indices of the sorted scores of documents which are the indices of the retrieved documents
    
    relevantRetrieved = [retrievedDocuments.index(i) for i in retrievedDocuments if i in relevantDocs]
    
    total=0
    numRelevantRetrieved=0
    for d in relevantRetrieved:
        numRelevantRetrieved=numRelevantRetrieved+1
        total=total+(numRelevantRetrieved/(d+1))
    
    return total/R

def MAP(documentsScores):
    total=0
    for index, qNum in enumerate(queries.keys()):
        total=total+AP(documentsScores[index], qNum)
    return total/get_Q() 




In [52]:
n1=len(λc_L)
print("n1={}".format(n1))


MeanAveragePrecisions=[]
numThreads=0
maxNumThreads=8
bestλc=-1
bestMap=-1
for i in range(0,n1):
    documentsScores = initializeDocumentsScores()
    T=[]
    queueDict={}
    for index,qNum in enumerate(queries.keys()):
        if numThreads<maxNumThreads :
            q = Queue()
            thread = Process(target = call_OkapiBM25_Docs, args =  (q,qNum,k1,b,k3,i, ) )
            T.append(thread)
            thread.start()
            queueDict[index]=q
            numThreads=numThreads+1
        else:
            for t, key in enumerate(queueDict): 
                documentsScores[key]=queueDict[key].get()
                T[t].join()
            T=[]
            queueDict={}
            numThreads=0
    for t, key in enumerate(queueDict): #for the the last threads there may be less than maxNumThreads
        documentsScores[key]=queueDict[key].get()
        T[t].join()
    T=[]
    queueDict={}
    numThreads=0
    path="/home/mosab/Desktop/IR/Ar/Project/results/"
    with open(path+"OkapiBM25Extended_λc="+str(λc_L[i])+" DocumentsScores.txt", "wb") as fp:
        pickle.dump(documentsScores, fp)
    MeanAveragePrecision=MAP(documentsScores)
    print("λc="+str(λc_L[i])+" MAP="+str(MeanAveragePrecision))
    if(MeanAveragePrecision>bestMap):
        bestMap=MeanAveragePrecision
        bestλc=λc_L[i]
    MeanAveragePrecisions.append(MeanAveragePrecision)
    with open(path+"OkapiBM25Extended_MAP Results.txt",'w') as f:
        f.write(str(MeanAveragePrecision)+": λc="+str(λc_L[i])+"\n")
    print("λc="+str(bestλc)+" MAP="+str(bestMap)+" Best")


path="/home/mosab/Desktop/IR/Ar/Project/results/"
MeanAveragePrecisions=[{"bestMAP":bestMap,"λc":bestλc}]+MeanAveragePrecisions
with open(path+"OkapiBM25Extended_MeanAveragePrecisions.txt", "wb") as fp:
    pickle.dump(MeanAveragePrecisions, fp)


print("==================================MAP is Done==================================")

n1=20
λc=0.1 MAP=0.41487951118213134
λc=0.1 MAP=0.41487951118213134 Best
λc=0.2 MAP=0.415247710851303
λc=0.2 MAP=0.415247710851303 Best
λc=0.30000000000000004 MAP=0.4160482978283488
λc=0.30000000000000004 MAP=0.4160482978283488 Best
λc=0.4 MAP=0.41711867782604645
λc=0.4 MAP=0.41711867782604645 Best
λc=0.5 MAP=0.41729380541964817
λc=0.5 MAP=0.41729380541964817 Best
λc=0.6 MAP=0.4174072655827343
λc=0.6 MAP=0.4174072655827343 Best
λc=0.7000000000000001 MAP=0.41832888761090875
λc=0.7000000000000001 MAP=0.41832888761090875 Best
λc=0.8 MAP=0.41882492234350627
λc=0.8 MAP=0.41882492234350627 Best
λc=0.9 MAP=0.41868005237820244
λc=0.8 MAP=0.41882492234350627 Best
λc=1.0 MAP=0.419616060172693
λc=1.0 MAP=0.419616060172693 Best
λc=1.1 MAP=0.41965352089150243
λc=1.1 MAP=0.41965352089150243 Best
λc=1.2000000000000002 MAP=0.4200034896935963
λc=1.2000000000000002 MAP=0.4200034896935963 Best
λc=1.3000000000000003 MAP=0.4209259946153721
λc=1.3000000000000003 MAP=0.4209259946153721 Best
λc=1.400000000000